# HCP Young Adult dataset analysis

## Dataset and project initialization

In [ ]:
!uv sync

In [ ]:
import glob, numpy as np, nibabel as nib, collections, pywt
import matplotlib.pyplot as plt
from pathlib import Path
from nibabel.cifti2.cifti2 import Cifti2Image
from scipy.signal import welch
from sklearn.model_selection import GroupKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from nilearn import plotting, image

In [ ]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
files = sorted(ROOT.glob("data/*/MNINonLinear/Results/tfMRI_*_??/*dtseries.nii"))

In [ ]:
DESIKAN = ["bankssts", "caudalanteriorcingulate", "caudalmiddlefrontal", None, "cuneus",
           "entorhinal", "fusiform", "inferiorparietal", "inferiortemporal",
           "isthmuscingulate", "lateraloccipital", "lateralorbitofrontal", "lingual",
           "medialorbitofrontal", "middletemporal", "parahippocampal", "paracentral",
           "parsopercularis", "parsorbitalis", "parstriangularis", "pericalcarine",
           "postcentral", "posteriorcingulate", "precentral", "precuneus",
           "rostralanteriorcingulate", "rostralmiddlefrontal", "superiorfrontal",
           "superiorparietal", "superiortemporal", "supramarginal", "frontalpole",
           "temporalpole", "transversetemporal", "insula"]      # None = corpus callosum

CODES = {c: f"{h}_{nm}" for k, nm in enumerate(DESIKAN) if nm
         for c, h in ((1001 + k, "L"), (2001 + k, "R"))}
CODES.update({8: "L_cerebellum", 10: "L_thalamus", 11: "L_caudate", 12: "L_putamen",
              13: "L_pallidum", 17: "L_hippocampus", 18: "L_amygdala", 26: "L_accumbens",
              28: "L_ventralDC", 47: "R_cerebellum", 49: "R_thalamus", 50: "R_caudate",
              51: "R_putamen", 52: "R_pallidum", 53: "R_hippocampus", 54: "R_amygdala",
              58: "R_accumbens", 60: "R_ventralDC", 16: "brainstem"})

CODE_LIST = sorted(CODES)
names = [CODES[c] for c in CODE_LIST]
R = len(CODE_LIST)
print(R, "regiona:", names[:3], "...", names[-3:])

In [ ]:
vol_files = sorted(ROOT.glob(
    "data/*/MNINonLinear/Results/tfMRI_*_??/*_hp0_clean_rclean_tclean.nii.gz"))
print(len(vol_files), "runova")
print(vol_files[0].parts[-5], vol_files[0].parts[-2])

In [ ]:
runs, meta, current = [], [], None

for f in vol_files:
    subject, task, enc = f.parts[-5], *f.parts[-2].split("_")[1:3]

    if subject != current:                          # parcelacija je po ispitaniku
        w = nib.load(str(ROOT / "data" / subject / "MNINonLinear/ROIs/wmparc.2.nii.gz"))
        codes = w.get_fdata().astype(np.int32).ravel()
        lab = np.zeros(codes.shape, dtype=np.int32)
        for i, c in enumerate(CODE_LIST, start=1):
            lab[codes == c] = i
        vox, lab_v = np.flatnonzero(lab), None
        lab_v = lab[vox]
        counts = np.bincount(lab_v, minlength=R + 1)[1:]
        current = subject

    img = nib.load(str(f))
    nt  = img.shape[3]
    x   = img.get_fdata(dtype=np.float32).reshape(-1, nt)[vox]
    runs.append(np.stack([np.bincount(lab_v, weights=x[:, j], minlength=R + 1)[1:] / counts
                          for j in range(nt)]).astype(np.float32))
    meta.append((subject, task, enc))
    print(f"{len(runs):3d}/{len(vol_files)}  {subject} {task}_{enc}  {runs[-1].shape}",
          flush=True)

## Pregled signala

In [ ]:
REGION       = next(i for i, n in enumerate(names) if n.endswith(REGION_NAME))
REGION_SHORT = names[REGION].replace("CIFTI_STRUCTURE_", "")

In [ ]:
print(len(names), names)

In [ ]:
subjects  = np.array([m[0] for m in meta])
encodings = np.array([m[2] for m in meta])
SUBJECTS, TASKS, ENCS = sorted(set(subjects)), sorted(set(y)), sorted(set(encodings))

print("ispitanici:", {i: s for i, s in enumerate(SUBJECTS)})
print("zadaci:    ", {i: t for i, t in enumerate(TASKS)})
print("kodiranja: ", {i: e for i, e in enumerate(ENCS)})

SUBJECT = SUBJECTS[2]        # 0 111211  1 135124  2 153126  3 192237  4 206525 ...
TASK    = TASKS[2]           # 0 EMOTION  1 GAMBLING  2 LANGUAGE  3 MOTOR
ENC     = ENCS[0]            # 4 RELATIONAL  5 SOCIAL  6 WM   |   0 LR  1 RL
IA, IB  = 0, 1               # indeksi uslova, vidi ispis "uslovi"
CZ      = 27                 # presek; MNI z = (CZ - 36) * 2 mm
DELAY   = 5.0
NFRAMES, NCOLS = 12, 4
REGION_NAME = names[REGION]

In [ ]:
ACCENT, MUTED = "#3b6fb6", "#9aa0a6"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.linewidth": 0.4, "grid.color": MUTED, "grid.alpha": 0.3,
    "axes.titlelocation": "left", "axes.titlesize": 10, "axes.labelsize": 10,
    "figure.constrained_layout.use": True,
})

In [ ]:
stem = f"tfMRI_{TASK}_{ENC}"
run  = ROOT / "data" / SUBJECT / "MNINonLinear/Results" / stem

REGION       = next(i for i, n in enumerate(names) if n.endswith(REGION_NAME))
REGION_SHORT = names[REGION].replace("CIFTI_STRUCTURE_", "")
RUNS_IDX     = np.flatnonzero((subjects == SUBJECT) & (y == TASK))

img4 = nib.load(str(run / f"{stem}_hp0_clean_rclean_tclean.nii.gz"))
TR   = float(img4.header.get_zooms()[3])
nx, ny, nz, nt = img4.shape
t    = np.arange(nt) * TR

sl       = np.asarray(img4.dataobj[:, :, CZ, :], dtype=np.float32)     # (nx, ny, nt)
mean_img = nib.load(str(run / f"{stem}_mean.nii.gz")).get_fdata()[:, :, CZ]
mask     = nib.load(str(run / "brainmask_fs.2.nii.gz")).get_fdata()[:, :, CZ] > 0
FRAMES   = np.linspace(0, nt - 1, NFRAMES).astype(int)

pct = np.where(mask[..., None],                                        # % promene
               (sl - mean_img[..., None]) / np.maximum(mean_img, 1)[..., None] * 100,
               np.nan)

CONDS = sorted(p.stem for p in (run / "EVs").glob("*.txt") if p.stem != "Sync")
COND_A, COND_B = CONDS[IA], CONDS[IB]

blocks = {}
for cond in (COND_A, COND_B):
    idx = set()
    for line in open(run / "EVs" / f"{cond}.txt"):
        onset, dur, _ = map(float, line.split())
        a = int(round((onset + DELAY) / TR))
        b = int(round((onset + dur + DELAY) / TR))
        idx |= set(range(max(a, 0), min(b, nt)))
    blocks[cond] = sorted(idx)

print(f"{SUBJECT} {stem}  nt={nt}  TR={TR}  z={CZ} (MNI z={(CZ - 36) * 2:+d} mm)  {REGION_SHORT}")
print("uslovi:", {i: c for i, c in enumerate(CONDS)})
print("frejmova:", {c: len(v) for c, v in blocks.items()})

In [ ]:
assert stem == f"tfMRI_{TASK}_{ENC}", "ponovo pokreni ćeliju A"

idx = np.flatnonzero((subjects == SUBJECT) & (y == TASK))        # LR i RL
fig, axes = plt.subplots(len(idx), 1, figsize=(13, 2.4 * len(idx)),
                         sharex=True, sharey=True, squeeze=False, layout="constrained")
for ax, i in zip(axes.flat, idx):
    sig = runs[i][:, REGION]
    sig = (sig - sig.mean()) / sig.std()
    ax.plot(np.arange(len(sig)) * TR, sig, lw=0.9, color=ACCENT)
    ax.set_title(f"{y[i]}_{encodings[i]}   ({len(sig)} frejmova, {len(sig) * TR:.0f} s)")
    ax.axhline(0, lw=0.6, color=MUTED, zorder=0)
axes.flat[-1].set_xlabel("vreme (s)")
fig.suptitle(f"{REGION_NAME} — {TASK}, ispitanik {SUBJECT}", x=0.01, ha="left")

In [ ]:
img4   = nib.load(str(run / f"{stem}_hp0_clean_rclean_tclean.nii.gz"))
vol    = img4.get_fdata(dtype=np.float32)                   # (91,109,91,nt), ~1 GB
mean3d = nib.load(str(run / f"{stem}_mean.nii.gz")).get_fdata()

diff3d     = vol[..., blocks[COND_A]].mean(-1) - vol[..., blocks[COND_B]].mean(-1)
contrast3d = np.where(mean3d > 0, diff3d / np.maximum(mean3d, 1) * 100, 0)
stat       = nib.Nifti1Image(contrast3d.astype(np.float32), img4.affine)

anat = ROOT / "data" / SUBJECT / "MNINonLinear/T1w_restore.2.nii.gz"
ttl  = f"{TASK}: {COND_A} − {COND_B} (%)"

plotting.plot_stat_map(stat, bg_img=str(anat), threshold=0.4, vmax=1.5,
                       display_mode="mosaic", title=ttl)


In [ ]:
i = np.flatnonzero((subjects == SUBJECT) & (y == TASK) & (encodings == ENC))[0]
Z = runs[i]
Z = ((Z - Z.mean(0)) / np.where(Z.std(0) > 0, Z.std(0), 1)).T          # (87, T), redosled = names

L = [k for k, n in enumerate(names) if n.startswith("L_")]
R = [k for k, n in enumerate(names) if n.startswith("R_")]

fig, axes = plt.subplots(2, 1, figsize=(16, 13), sharex=True, layout="constrained")
for ax, idx, side in [(axes[0], L, "levo"), (axes[1], R, "desno")]:
    im = ax.imshow(Z[idx], aspect="auto", cmap="RdBu_r", vmin=-2.5, vmax=2.5,
                   extent=[0, Z.shape[1] * TR, len(idx), 0], interpolation="nearest")
    ax.set_yticks(np.arange(len(idx)) + 0.5)
    ax.set_yticklabels([names[k] for k in idx], fontsize=7)
    ax.set_xlabel("vreme (s)")
    ax.set_title(side)
    ax.grid(False)
    axes[0].set_xlabel("")          # samo donji panel nosi oznaku
    for f in blocks[COND_A]:
        ax.axvline(f * TR, color="k", lw=0.3, alpha=0.15)
fig.colorbar(im, ax=axes, shrink=0.6, label="z")
fig.suptitle(f"{SUBJECT} — {stem}, z-score po regionu", x=0.01, ha="left")

In [ ]:
assert stem == f"tfMRI_{TASK}_{ENC}", "ponovo pokreni ćeliju A"

sd = sl.std(axis=-1)
vx, vy = np.unravel_index(np.argmax(np.where(mask, sd, 0)), sd.shape)

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for ax, sig, title in [(axes[0], sl[vx, vy, :], f"voksel ({vx}, {vy}, {CZ})"),
                       (axes[1], sl[mask].mean(axis=0), f"prosek preseka z={CZ}")]:
    ax.plot(t, (sig - sig.mean()) / sig.std(), lw=0.9, color=ACCENT)
    ax.set_title(title)
    ax.axhline(0, lw=0.6, color=MUTED, zorder=0)
axes[-1].set_xlabel("vreme (s)")
fig.suptitle(f"{SUBJECT} — {stem}", x=0.01, ha="left")

In [ ]:
NPERSEG = 128
FS, NYQ = 1 / TR, 1 / (2 * TR)

psd = {}
for task in TASKS:
    acc = []
    for i in np.flatnonzero(y == task):
        sig = runs[i][:, REGION]
        f, p = welch((sig - sig.mean()) / sig.std(), fs=FS, nperseg=NPERSEG)
        acc.append(p)
    psd[task] = np.mean(acc, axis=0)
grand = np.mean(list(psd.values()), axis=0)

NR = -(-len(TASKS) // NCOLS)
fig, axes = plt.subplots(NR, NCOLS, figsize=(3.3 * NCOLS, 2.9 * NR),
                         sharex=True, sharey=True)
for ax, task in zip(axes.flat, TASKS):
    for l in range(1, 6):
        ax.axvline(NYQ / 2 ** l, lw=0.5, color=MUTED, alpha=0.6, zorder=0)
    ax.semilogy(f, grand, lw=1.0, color=MUTED, zorder=1)
    ax.semilogy(f, psd[task], lw=1.5, color=ACCENT, zorder=2)
    ax.set_title(f"{task}  (n={np.sum(y == task)})")
for j, ax in enumerate(axes.flat):
    if j >= len(TASKS):
        ax.set_visible(False)
    elif j + NCOLS >= len(TASKS):
        ax.set_xlabel("frekvencija (Hz)")
fig.suptitle(f"Spektar po zadatku — {REGION_NAME} (sivo = prosek svih)", x=0.01, ha="left")

## Obrada podataka

In [ ]:
FS = 1 / TR
f, P = welch(Xw, fs=FS, nperseg=WIN, axis=1)          # (n, nf, 87)
p = P / P.sum(axis=1, keepdims=True)                  # relativna snaga
NLEV = 4                                    # isti broj nivoa kao DWT
NYQ  = 1 / (2 * TR)
edges = [f[1]] + [NYQ / 2 ** k for k in range(NLEV, 0, -1)] + [NYQ]
BANDS = list(zip(edges[:-1], edges[1:]))
BAND_NAMES = [f"A{NLEV}"] + [f"D{k}" for k in range(NLEV, 0, -1)]

for nm, (lo, hi) in zip(BAND_NAMES, BANDS):
    print(f"  {nm:3s}  {lo:.4f}–{hi:.4f} Hz")

In [ ]:
blocks = [p[:, (f >= lo) & (f < hi), :].sum(axis=1) for lo, hi in BANDS]
blocks += [-(p * np.log(p + 1e-12)).sum(axis=1),      # spektralna entropija
           (f[None, :, None] * p).sum(axis=1),        # spektralno težište
           (Xw[:, :-1] * Xw[:, 1:]).mean(axis=1),     # autokorelacija lag 1
           (Xw[:, :-5] * Xw[:, 5:]).mean(axis=1)]     # autokorelacija lag 5

BLOCK_NAMES = [f"band_{nm}" for nm in BAND_NAMES] + ["entropy", "centroid", "ac1", "ac5"]
FEAT_NAMES  = [f"{b}:{r}" for b in BLOCK_NAMES for r in names]

print(f"F_std: {F_std.shape}  ({len(BLOCK_NAMES)} tipa × {len(names)} regiona)")
print(f"NaN/inf: {(~np.isfinite(F_std)).sum()}")

In [ ]:
WAVELET = "db4"                       # ili "sym4"
L   = pywt.dwt_max_level(WIN, pywt.Wavelet(WAVELET).dec_len)
NYQ = 1 / (2 * TR)

coeffs = pywt.wavedec(Xw, WAVELET, level=L, axis=1)      # [cA_L, cD_L, ..., cD_1]
SUB    = [f"A{L}"] + [f"D{k}" for k in range(L, 0, -1)]

print(f"{WAVELET}, L = {L}   (filtar {pywt.Wavelet(WAVELET).dec_len} koef., prozor {WIN})")
for s, c in zip(SUB, coeffs):
    k = int(s[1:])
    lo, hi = (0, NYQ / 2 ** L) if s[0] == "A" else (NYQ / 2 ** k, NYQ / 2 ** (k - 1))
    print(f"  {s:3s}  {c.shape[1]:3d} koeficijenata   {lo:.3f}–{hi:.3f} Hz")

In [ ]:
energy = [(c ** 2).sum(axis=1) for c in coeffs]              # (n, 87) po podopsegu
total  = np.maximum(np.sum(energy, axis=0), 1e-12)
rel    = [e / total for e in energy]                          # relativna energija

ent = []
for c in coeffs:                                              # entropija koeficijenata
    q = c ** 2
    q = q / np.maximum(q.sum(axis=1, keepdims=True), 1e-12)
    ent.append(-(q * np.log(q + 1e-12)).sum(axis=1))

F_dwt     = np.concatenate(rel + ent, axis=1).astype(np.float32)
DWT_NAMES = [f"{p}_{s}:{r}" for p in ("E", "H") for s in SUB for r in names]
F_all     = np.concatenate([F_std, F_dwt], axis=1)
ALL_NAMES = FEAT_NAMES + DWT_NAMES

print(f"standardna {F_std.shape}   DWT {F_dwt.shape}   zajedno {F_all.shape}")
print(f"NaN/inf: {(~np.isfinite(F_dwt)).sum()}")

In [ ]:
WIN, STRIDE = 176, 88                      # 176 = najkraći run; 50% preklapanja
N_TRAIN, N_VAL, N_TEST, SEED = 6, 2, 2, 0

Xw, yw, gw, srcw = [], [], [], []
for i, r in enumerate(runs):
    starts = list(range(0, r.shape[0] - WIN + 1, STRIDE))
    if starts[-1] != r.shape[0] - WIN:
        starts.append(r.shape[0] - WIN)                    # rep runa
    for s in starts:
        seg = r[s:s + WIN]
        Xw.append((seg - seg.mean(0)) / np.where(seg.std(0) > 0, seg.std(0), 1))
        yw.append(y[i]); gw.append(subjects[i]); srcw.append((i, s))

Xw   = np.stack(Xw).astype(np.float32)                     # (357, 176, 87)
yw, gw, srcw = np.array(yw), np.array(gw), np.array(srcw)

perm = np.random.default_rng(SEED).permutation(np.array(SUBJECTS))
SUBJ = {"train": sorted(perm[:N_TRAIN]),
        "val":   sorted(perm[N_TRAIN:N_TRAIN + N_VAL]),
        "test":  sorted(perm[N_TRAIN + N_VAL:])}
IDX  = {k: np.flatnonzero(np.isin(gw, s)) for k, s in SUBJ.items()}

LOSO      = list(LeaveOneGroupOut().split(Xw, yw, gw))
LOSO_SUBJ = [gw[te][0] for _, te in LOSO]

assert sum(len(v) for v in IDX.values()) == len(Xw)
print(f"prozora: {len(Xw)}  oblik {Xw.shape}")
for k in ("train", "val", "test"):
    print(f"  {k:5s} {list(SUBJ[k])} -> {len(IDX[k]):3d}")
print(f"LOSO: {len(LOSO)} foldova")

In [ ]:
print(f"{'skup':20s} {'val':>6s}   {'LOSO':>14s}   opseg")
for tag, F in [("A standardna", F_std), ("B standardna+DWT", F_all), ("C samo DWT", F_dwt)]:
    v = GaussianNB().fit(F[IDX["train"]], yw[IDX["train"]]).score(F[IDX["val"]], yw[IDX["val"]])
    a = np.array([GaussianNB().fit(F[tr], yw[tr]).score(F[te], yw[te]) for tr, te in LOSO])
    print(f"{tag:20s} {v:6.3f}   {a.mean():6.3f} ± {a.std():.3f}   {a.min():.2f}–{a.max():.2f}")
print(f"{'šansa':20s} {1/len(set(yw)):6.3f}")

In [ ]:
sub_acc = {}
for k, s in enumerate(SUB):
    Fk = np.concatenate([rel[k], ent[k]], axis=1).astype(np.float32)
    a  = np.array([GaussianNB().fit(Fk[tr], yw[tr]).score(Fk[te], yw[te]) for tr, te in LOSO])
    sub_acc[s] = a
    print(f"  {s:3s}  LOSO {a.mean():.3f} ± {a.std():.3f}")

fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
m = [sub_acc[s].mean() for s in SUB]
e = [sub_acc[s].std() for s in SUB]
ax.errorbar(range(len(SUB)), m, yerr=e, marker="o", ms=7, lw=2, color=ACCENT, capsize=4)
ax.axhline(1 / len(set(yw)), lw=1, color=MUTED, ls="--")
ax.set_xticks(range(len(SUB)))
ax.set_xticklabels(SUB)
ax.set_ylabel("tačnost (LOSO)")
ax.set_xlabel("podopseg")
ax.set_title("Tačnost po podopsegu — isprekidano: slučajno pogađanje")

In [ ]:
F = F_all                                        # skup B
pred = np.empty_like(yw)
for tr, te in LOSO:
    pred[te] = GaussianNB().fit(F[tr], yw[tr]).predict(F[te])

labels = sorted(set(yw))
cm = confusion_matrix(yw, pred, labels=labels, normalize="true")

fig, ax = plt.subplots(figsize=(6.5, 6), layout="constrained")
ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, cmap="Blues",
                                                       colorbar=False, values_format=".2f")
ax.set_xlabel("predviđeno"); ax.set_ylabel("stvarno")
ax.set_title(f"LOSO, skup B — ukupno {(pred == yw).mean():.3f}")
plt.xticks(rotation=45, ha="right")
ax.grid(False)

In [ ]:
Fv, pv = f_classif(F_all[IDX["train"]], yw[IDX["train"]])
Fv = np.nan_to_num(Fv)
order = np.argsort(-Fv)

print("najdiskriminativnijih 25:")
for j in order[:25]:
    print(f"  {ALL_NAMES[j]:34s} F={Fv[j]:7.2f}")

# udeo po tipu obeležja i po podopsegu
import re
kind = np.array([n.split(":")[0].split("_")[0] for n in ALL_NAMES])
top  = order[:200]
print("\nudeo u top 200 po tipu:", dict(sorted(collections.Counter(kind[top]).items(),
                                               key=lambda t: -t[1])))
reg  = np.array([n.split(":")[1] for n in ALL_NAMES])
print("najčešći regioni:", collections.Counter(reg[top]).most_common(10))

In [ ]:
a_std = np.array([GaussianNB().fit(F_std[tr], yw[tr]).score(F_std[te], yw[te]) for tr, te in LOSO])
a_all = np.array([GaussianNB().fit(F_all[tr], yw[tr]).score(F_all[te], yw[te]) for tr, te in LOSO])
d = a_all - a_std
from scipy.stats import wilcoxon
print(f"razlika po ispitaniku: {d.mean():+.3f} ± {d.std():.3f}   pozitivnih {(d>0).sum()}/10")
print(wilcoxon(a_all, a_std))

In [ ]:
F_nob = np.concatenate(blocks[len(BANDS):], axis=1).astype(np.float32)   # bez band_*
F_nob_dwt = np.concatenate([F_nob, F_dwt], axis=1)

res = {}
for tag, F in [("bez band_*", F_nob), ("bez band_* + DWT", F_nob_dwt),
               ("A standardna", F_std), ("B standardna+DWT", F_all)]:
    res[tag] = np.array([GaussianNB().fit(F[tr], yw[tr]).score(F[te], yw[te])
                         for tr, te in LOSO])
    print(f"{tag:20s} {F.shape[1]:5d} obeležja   LOSO {res[tag].mean():.3f} ± {res[tag].std():.3f}")

for a, b in [("bez band_*", "bez band_* + DWT"), ("A standardna", "B standardna+DWT")]:
    d = res[b] - res[a]
    print(f"\n{b} − {a}: {d.mean():+.3f} ± {d.std():.3f}  pozitivnih {(d>0).sum()}/10  "
          f"p={wilcoxon(res[b], res[a]).pvalue:.3f}")

In [ ]:
loc = []
for c in coeffs:                                   # c: (n, len_k, 87)
    a = np.abs(c)
    q = np.sort(c ** 2, axis=1)[:, ::-1, :]        # energija, opadajuće
    k = max(1, int(0.1 * c.shape[1]))
    loc += [a.var(axis=1),                                        # rasipanje |koef.|
            a.max(axis=1),                                        # najveći koeficijent
            q[:, :k, :].sum(axis=1) / np.maximum(q.sum(axis=1), 1e-12)]   # koncentracija

F_loc  = np.concatenate(loc, axis=1).astype(np.float32)
F_all2 = np.concatenate([F_std, F_loc], axis=1)
LOC_NAMES = [f"{p}_{s}:{r}" for s in SUB for p in ("V", "M", "K") for r in names]

for tag, F in [("A standardna", F_std), ("B2 standardna+DWT_lok", F_all2)]:
    res[tag] = np.array([GaussianNB().fit(F[tr], yw[tr]).score(F[te], yw[te])
                         for tr, te in LOSO])
    print(f"{tag:24s} {F.shape[1]:5d}   LOSO {res[tag].mean():.3f} ± {res[tag].std():.3f}")
d = res["B2 standardna+DWT_lok"] - res["A standardna"]
print(f"razlika: {d.mean():+.3f} ± {d.std():.3f}  pozitivnih {(d>0).sum()}/10  "
      f"p={wilcoxon(res['B2 standardna+DWT_lok'], res['A standardna']).pvalue:.3f}")

In [ ]:
lr = lambda: make_pipeline(StandardScaler(),
                           LogisticRegression(max_iter=2000, C=0.1))   # C nizak: p >> n

for tag, F in [("A standardna", F_std), ("B standardna+DWT", F_all)]:
    a = np.array([lr().fit(F[tr], yw[tr]).score(F[te], yw[te]) for tr, te in LOSO])
    res["LR " + tag] = a
    print(f"LR {tag:20s} LOSO {a.mean():.3f} ± {a.std():.3f}   opseg {a.min():.2f}–{a.max():.2f}")

d = res["LR B standardna+DWT"] - res["LR A standardna"]
print(f"\nLR razlika: {d.mean():+.3f} ± {d.std():.3f}  pozitivnih {(d>0).sum()}/10  "
      f"p={wilcoxon(res['LR B standardna+DWT'], res['LR A standardna']).pvalue:.3f}")